In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, precision_recall_curve

def save_answer(filename, answer_str):
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(str(answer_str).strip())
    print(f"Ответ для '{filename}' сохранен: {answer_str}")

df_cls = pd.read_csv('datasets/classification.csv')
tn, fp, fn, tp = confusion_matrix(df_cls['true'], df_cls['pred']).ravel()
save_answer('metrics_ans1.txt', f"{tp} {fp} {fn} {tn}")

acc = accuracy_score(df_cls['true'], df_cls['pred'])
prec = precision_score(df_cls['true'], df_cls['pred'])
rec = recall_score(df_cls['true'], df_cls['pred'])
f1 = f1_score(df_cls['true'], df_cls['pred'])
save_answer('metrics_ans2.txt', f"{acc:.2f} {prec:.2f} {rec:.2f} {f1:.2f}")

df_scores = pd.read_csv('datasets/scores.csv')
y_true = df_scores['true']
classifiers = ['score_logreg', 'score_svm', 'score_knn', 'score_tree']

auc_results = {clf: roc_auc_score(y_true, df_scores[clf]) for clf in classifiers}
best_auc_clf = max(auc_results, key=auc_results.get)
save_answer('metrics_ans3.txt', best_auc_clf)

precision_active_results = {}
for clf in classifiers:
    precisions, recalls, _ = precision_recall_curve(y_true, df_scores[clf])
    precision_active_results[clf] = np.max(precisions[recalls >= 0.7])
best_pr_clf = max(precision_active_results, key=precision_active_results.get)
save_answer('metrics_ans4.txt', best_pr_clf)

print("\n--- Проверка расчетов ---")
print(f"Матрица ошибок (TP FP FN TN): {tp} {fp} {fn} {tn}")
print(f"Метрики (Acc Prec Rec F1): {acc:.2f} {prec:.2f} {rec:.2f} {f1:.2f}")
print("\nЗначения AUC-ROC:")
for k, v in auc_results.items():
    print(f"  {k}: {v:.4f}")
print("\nМакс. Precision при Recall >= 70%:")
for k, v in precision_active_results.items():
    print(f"  {k}: {v:.4f}")

Ответ для 'metrics_ans1.txt' сохранен: 43 34 59 64
Ответ для 'metrics_ans2.txt' сохранен: 0.54 0.56 0.42 0.48
Ответ для 'metrics_ans3.txt' сохранен: score_logreg
Ответ для 'metrics_ans4.txt' сохранен: score_tree

--- Проверка расчетов ---
Матрица ошибок (TP FP FN TN): 43 34 59 64
Метрики (Acc Prec Rec F1): 0.54 0.56 0.42 0.48

Значения AUC-ROC:
  score_logreg: 0.7192
  score_svm: 0.7087
  score_knn: 0.6352
  score_tree: 0.6919

Макс. Precision при Recall >= 70%:
  score_logreg: 0.6303
  score_svm: 0.6228
  score_knn: 0.6066
  score_tree: 0.6518
